# FROSTWATCH V5 — PHASE E: 3-ARM GRAND ENSEMBLE (V2 + V5C + V5S) — **PRODUCTION = SUBMISSION A**

**What it does (NO TRAINING, ~1h):** regenerates the team-best submission
config (A, 16.27 AP). Same 3-arm mean fusion, same extraction:

- **ARM A (V2):** 5 checkpoints from `V2 WEIGHT/` — 8ch, 384x384 (LB AP 13.88, AP50/large floor)
- **ARM B (V5C):** 2 checkpoints from Phase C Session 2 — corrected 13ch, 512 (boundary quality, AP75 moat)
- **ARM C (V5S):** `ep100`+`ep090` from V5S_RUN1 — MAnet/mit_b4, 13ch, 512 (**small-object specialist**)

**PRODUCTION CONFIG (Submission A, 16.27 best):** `W_V2=0.2, W_V5C=0.5,
W_V5S=0.3`; extraction at production (`THR_LO=0.25, THR_HI=0.50,
MIN_AREA=5, TOP_K=10`).

**Submission history (all axes closed at this level):**
- A (08-30) = **16.27 BEST** — this notebook reproduces it
- B (08-31, weights 0.15/0.55/0.3) = 16.08 (−0.19) — weights axis CLOSED
- C (08-31 15:27, ARM C → Run-2 Copy-Paste ckpts) = 15.73 (−0.54) — the
  Run-2 harness win (+1.51 clean holdout) did NOT transfer to the LB; only
  its small/boundary signature did (AP75 10.91 new best, APs +0.24); AP50
  −2.19. ARM C reverted to Run 1; Copy-Paste axis CLOSED at the arm level.

**How to run:**
1. New Kaggle notebook | GPU: T4, with **4 inputs**:
   - Your dataset `frostwatch-v2-weights`
   - **Notebooks → your PHASE C SESSION 2 notebook output** (`v5c_checkpoints/*.pth`)
   - **Notebooks → your V5S_RUN1 notebook output** (`v5s_checkpoints/*.pth`)
   - The competition dataset (or let it auto-download)
2. **File → Import Notebook** (this file)
3. **Save Version → Save & Run All**
4. Download `FrostWatch.json` from Output → submit to the HF space

**VERIFY ON STARTUP LOG (all three must be full):**
`[V5-E] ARM A: found 5/5` | `[V5-E] ARM B: found 2/2` | `[V5-E] ARM C (V5S): found 2/2`
If any arm is short, a loud warning prints — stop and check the inputs.

DO NOT EDIT ANYTHING. Zero configuration.

In [ ]:
# ============================================================================
#  FROSTWATCH V5 — PHASE E: 3-ARM GRAND ENSEMBLE (V2 + V5C + V5S) — PRODUCTION
#  REVIEW.md v2.0 Task 6 — INFERENCE ONLY
#
#  PRODUCTION CONFIG = SUBMISSION A (08-30, AP 16.27 — TEAM BEST, Rank #2):
#    W_V2=0.2, W_V5C=0.5, W_V5S=0.3  (arm weights sum to 1.0)
#    ARM C = V5S RUN-1 ep100+ep090 (v5s_manet_mitb4_*, V5S_RUN1 output)
#  Submission history:
#    A (08-30, Run-1 arm, 0.2/0.5/0.3) = 16.27 BEST
#    B (08-31, weights 0.15/0.55/0.3) = 16.08 (-0.19) — weights axis CLOSED
#    C (08-31 15:27, ARM C -> Run-2 Copy-Paste ckpts, weights A) = 15.73
#        (-0.54) — the Run-2 harness win (+1.51 clean holdout) did NOT
#        transfer; only its small/boundary signature did (AP75 10.91 new
#        best, APs +0.24); AP50 cratered -2.19. ARM C REVERTED to Run 1.
#        Copy-Paste axis CLOSED at the arm level.
#
#    ARM A (V2):  5 models, 8ch, 384x384 anisotropic, 5-pass TTA
#    ARM B (V5C): 2 models, corrected 13ch, 512 pad-square, 8-fold TTA
#    ARM C (V5S): 2 ckpts (ep100+ep090, arm-mean), MAnet/mit_b4,
#                 corrected 13ch, 512 pad-square, 8-fold TTA (RUN 1)
#
#  Extraction: Phase C dual-threshold at PRODUCTION config (unchanged):
#    THR_LO=0.25, THR_HI=0.50, MIN_AREA=5, TOP_K=10, score=0.5*mean+0.5*p95
#  (THR_HI sweep closed 08-29: 0.45 -> 15.64, 0.55 -> 15.52; 0.50 optimum.)
#
#  NO TRAINING. ~1h on a T4.
#
#  HOW TO RUN (Kaggle):
#    1. New Kaggle notebook | GPU: T4, with inputs:
#       - Your dataset "frostwatch-v2-weights" (5 .pth)
#       - Notebooks -> your PHASE C SESSION 2 notebook (v5c_checkpoints/*.pth)
#       - Notebooks -> your V5S_RUN1 notebook (v5s_checkpoints/*.pth)
#       - The competition dataset (or let it auto-download)
#    2. File -> Import Notebook (this file)
#    3. Save Version -> Save & Run All
#    4. Download /kaggle/working/FrostWatch.json -> submit to HF space.
#
#  VERIFY ON STARTUP LOG (must appear before inference starts):
#    [V5-E] ARM A: found 5/5 V2 checkpoints ...
#    [V5-E] ARM B: found 2/2 V5C checkpoints ...
#    [V5-E] ARM C (V5S): found 2/2 checkpoints ...
#  If any arm is short, a loud warning prints — STOP and check inputs.
#
#  DO NOT EDIT ANYTHING BELOW. Zero configuration.
# ============================================================================
import os
import sys
import json
import glob
import time
import random
import warnings
from collections import OrderedDict

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from scipy import ndimage

torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

try:
    import segmentation_models_pytorch as smp
    from pycocotools import mask as mask_utils
except ImportError:
    print("[V5-E] Auto-installing required packages...")
    os.system("pip install -q segmentation-models-pytorch pycocotools "
              "huggingface_hub timm scipy pandas")
    import segmentation_models_pytorch as smp
    from pycocotools import mask as mask_utils

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1: DATA DISCOVERY
# ════════════════════════════════════════════════════════════════════════════

DATA_ROOT_CANDIDATES = [
    "/kaggle/input/2026geoaiarcticchallengedataset/competition_release",
    "/kaggle/input/2026geoaiarcticchallengedataset",
    "/kaggle/working/dataset/competition_release",
    "/kaggle/working/dataset",
    "/content/dataset",
]

def find_data_dir():
    for cand in DATA_ROOT_CANDIDATES:
        if os.path.exists(cand):
            found = glob.glob(f"{cand}/**/instances_train.json", recursive=True)
            if found:
                return os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
    found = (glob.glob("/kaggle/input/**/instances_train.json", recursive=True) +
             glob.glob("/kaggle/working/**/instances_train.json", recursive=True))
    if found:
        return os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
    return None

DATA_DIR = find_data_dir()
if DATA_DIR is None:
    print("[V5-E] Dataset not found locally. Auto-downloading from Hugging Face Hub...")
    import zipfile
    from huggingface_hub import snapshot_download
    RAW_DL_DIR = "/kaggle/working/dataset" if os.path.exists("/kaggle") else "./dataset"
    snapshot_download(repo_id="cyber2a/2026GeoAIArcticChallengeDataset",
                      repo_type="dataset", local_dir=RAW_DL_DIR)
    for zf in glob.glob(f"{RAW_DL_DIR}/**/*.zip", recursive=True):
        with zipfile.ZipFile(zf, 'r') as z:
            z.extractall(os.path.dirname(zf))
    found = glob.glob(f"{RAW_DL_DIR}/**/instances_train.json", recursive=True)
    assert found, "[V5-E] Auto-download failed."
    DATA_DIR = os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
assert DATA_DIR is not None, "Attach the competition dataset as a notebook input."
TEST_IMG_DIR = os.path.join(DATA_DIR, "test", "images")
TEST_MANIFEST = glob.glob(f"{DATA_DIR}/**/test_manifest.csv", recursive=True)[0]
print(f"[V5-E] Dataset root: {DATA_DIR}")

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "./v5_phase_e"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 2: CHECKPOINT DISCOVERY (all three arms, from ANY attached input)
# ════════════════════════════════════════════════════════════════════════════

V2_REGISTRY = OrderedDict({
    "unetpp_resnet34": {"arch": "UnetPlusPlus", "encoder": "resnet34"},
    "fpn_mit_b3":      {"arch": "FPN",          "encoder": "mit_b3"},
    "unet_resnet34":   {"arch": "Unet",         "encoder": "resnet34"},
    "linknet_resnet34": {"arch": "Linknet",     "encoder": "resnet34"},
    "unet_mit_b2":     {"arch": "Unet",         "encoder": "mit_b2"},
})

# Run-1 harness winners (08-29): ep100 #1 (AP 50.88/AP75 54.73/APs 40.53),
# ep090 #2 (50.74). Arm = mean of the two, mirroring the V5C arm structure.
# (Run-2 v5s2_* swap tried as Submission C: LB 15.73, DOWN -0.54 — reverted.)
V5S_TAGS = ["ep100", "ep090"]

def find_checkpoints():
    """Search every attached input for the checkpoints of all three arms."""
    all_pth = glob.glob("/kaggle/input/**/*.pth", recursive=True)
    v2_found, v5c_found, v5s_found = OrderedDict(), OrderedDict(), OrderedDict()
    for p in all_pth:
        base = os.path.basename(p)
        stem = base[:-4] if base.endswith(".pth") else base
        for name in V2_REGISTRY:
            if name in stem and stem.startswith(name):
                v2_found.setdefault(name, p)
        for name in ("fpn_mit_b5", "unet_mit_b3"):
            if stem in (f"v5c_{name}_final", f"v5c_{name}_last") and name not in v5c_found:
                v5c_found[name] = p
        for tag in V5S_TAGS:
            if stem == f"v5s_manet_mitb4_{tag}" and tag not in v5s_found:
                v5s_found[tag] = p
    return v2_found, v5c_found, v5s_found

V2_CKPTS, V5C_CKPTS, V5S_CKPTS = find_checkpoints()

V2_MODEL_LIST = [(n, p) for n, p in V2_CKPTS.items()]
print(f"[V5-E] ARM A (V2):  found {len(V2_MODEL_LIST)}/5 checkpoints: "
      f"{[n for n, _ in V2_MODEL_LIST]}")
print(f"[V5-E] ARM B (V5C): found {len(V5C_CKPTS)}/2 checkpoints: "
      f"{sorted(V5C_CKPTS)}")
print(f"[V5-E] ARM C (V5S): found {len(V5S_CKPTS)}/2 checkpoints: "
      f"{sorted(V5S_CKPTS)} (want {V5S_TAGS})")
if len(V2_MODEL_LIST) < 5:
    print("[V5-E] !! WARNING: ARM A incomplete — the 'V2 WEIGHT' dataset is "
          "not fully attached. Continuing with the models found.")
if len(V5C_CKPTS) < 2:
    print("[V5-E] !! WARNING: ARM B incomplete — attach the Phase C Session 2 "
          "notebook output (v5c_checkpoints). Continuing with what was found.")
if len(V5S_CKPTS) < 2:
    print("[V5-E] !! WARNING: ARM C incomplete — attach the V5S_RUN1 notebook "
          "output (v5s_checkpoints). Continuing with what was found.")
assert V2_MODEL_LIST or V5C_CKPTS or V5S_CKPTS, "[V5-E] STOP: no checkpoints found at all."

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 3: SHARED PREPROCESSING + ARM-SPECIFIC PATHS
# ════════════════════════════════════════════════════════════════════════════

def load_npz(path):
    img = np.load(path)['image'].astype(np.float32)
    return np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

# ── ARM A: V2 preprocessing (anisotropic 384x384, 8ch, per-band clip/z) ────
V2_IMG_SIZE = 384

def preprocess_v2(img8):
    arr = cv2.resize(img8, (V2_IMG_SIZE, V2_IMG_SIZE),
                     interpolation=cv2.INTER_LINEAR)
    for c in range(arr.shape[2]):
        band = arr[:, :, c]
        p2, p98 = np.percentile(band, 2), np.percentile(band, 98)
        if p98 - p2 > 1e-8:
            band = np.clip(band, p2, p98)
        mu, sigma = band.mean(), band.std() + 1e-8
        arr[:, :, c] = (band - mu) / sigma
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return torch.from_numpy(arr.transpose(2, 0, 1))

@torch.no_grad()
def predict_v2(model, tensor8):
    """V2's exact TTA: 4 rotations + horizontal flip (5 passes)."""
    preds = []
    batch = tensor8.unsqueeze(0).to(DEVICE)
    for k in range(4):
        rotated = torch.rot90(batch, k, dims=[2, 3])
        prob = torch.sigmoid(model(rotated))
        prob = torch.rot90(prob, -k, dims=[2, 3])
        preds.append(prob.squeeze().cpu().numpy())
    flipped = torch.flip(batch, dims=[3])
    prob_f = torch.sigmoid(model(flipped))
    prob_f = torch.flip(prob_f, dims=[3])
    preds.append(prob_f.squeeze().cpu().numpy())
    return np.mean(preds, axis=0)

# ── ARM B + C: 13ch preprocessing (corrected tensor, pad-square 512) ────────
V5C_IMG_SIZE = 512

def compute_channels13(img8):
    eps = 1e-8
    red = img8[:, :, 0].astype(np.float32)
    green = img8[:, :, 1].astype(np.float32)
    blue = img8[:, :, 2].astype(np.float32)
    ndvi = img8[:, :, 3].astype(np.float32)
    rele = img8[:, :, 4].astype(np.float32)

    der = np.zeros(img8.shape[:2] + (5,), dtype=np.float32)
    der[:, :, 0] = (red - blue) / (red + blue + eps)
    der[:, :, 1] = (red + green + blue) / 3.0
    gy = cv2.Sobel(rele, cv2.CV_32F, 1, 0, ksize=3)
    gx = cv2.Sobel(rele, cv2.CV_32F, 0, 1, ksize=3)
    slope = np.sqrt(gx * gx + gy * gy)
    der[:, :, 2] = slope
    der[:, :, 3] = cv2.Laplacian(rele, cv2.CV_32F)
    der[:, :, 4] = ndvi * slope
    return np.concatenate([img8.astype(np.float32), der], axis=-1)

def clip_zscore(img):
    for c in range(img.shape[2]):
        band = img[:, :, c]
        p2, p98 = np.percentile(band, 2), np.percentile(band, 98)
        if p98 - p2 > 1e-8:
            band = np.clip(band, p2, p98)
        mu, sigma = band.mean(), band.std() + 1e-8
        img[:, :, c] = (band - mu) / sigma
    return img

def preprocess_v5c(img8):
    img13 = compute_channels13(img8)
    img13 = clip_zscore(img13)
    img13 = np.nan_to_num(img13, nan=0.0, posinf=0.0, neginf=0.0)
    H, W = img13.shape[:2]
    s = max(H, W)
    sq = np.zeros((s, s, 13), dtype=np.float32)
    sq[:H, :W] = img13
    if s != V5C_IMG_SIZE:
        sq = cv2.resize(sq, (V5C_IMG_SIZE, V5C_IMG_SIZE),
                        interpolation=cv2.INTER_LINEAR)
    return torch.from_numpy(sq.transpose(2, 0, 1)[np.newaxis]), (H, W)

@torch.no_grad()
def predict_v5c(model, x):
    """13ch arms' TTA: 8-fold dihedral (V5C's exact recipe; V5S identical)."""
    model.eval()
    probs = []
    for k in range(4):
        for flip in (False, True):
            y = torch.rot90(x, k, dims=(2, 3))
            if flip:
                y = torch.flip(y, dims=(3,))
            p = torch.sigmoid(model(y.to(DEVICE)))
            if flip:
                p = torch.flip(p, dims=(3,))
            p = torch.rot90(p, -k, dims=(2, 3))
            probs.append(p.cpu())
    return torch.stack(probs).mean(0)

def prob_to_native(prob512, native_hw):
    """13ch arms' exact pad-crop + native resize."""
    H, W = native_hw
    S = prob512.shape[0]
    s = max(H, W)
    ch, cw = max(1, int(round(H / s * S))), max(1, int(round(W / s * S)))
    prob = prob512[:ch, :cw]
    return cv2.resize(prob, (W, H), interpolation=cv2.INTER_LINEAR)

# ── MERGE + EXTRACTION (Phase C dual-threshold, PRODUCTION config) ──────────
# PRODUCTION = Submission A (16.27 best): weights 0.2/0.5/0.3, ARM C = Run-1.
# (B 0.15/0.55/0.3 = 16.08, weights axis CLOSED. C Run-2 arm = 15.73, DOWN
# -0.54 — Copy-Paste axis CLOSED at the arm level; reverted 08-31.)
# Extraction thresholds stay at production (THR_HI sweep closed 08-29:
# 0.45 -> 15.64, 0.55 -> 15.52; 0.50 is the optimum).
FUSION = "mean"     # 3-arm weighted mean ("max" closed 08-28, down -2.59)
W_V2 = 0.2
W_V5C = 0.5
W_V5S = 0.3
THR_LO = 0.25
THR_HI = 0.50
MIN_AREA = 5
TOP_K = 10

def instances_from_prob(prob):
    binary = (prob > THR_LO).astype(np.uint8)
    labeled, n = ndimage.label(binary, structure=np.ones((3, 3), dtype=int))
    out = []
    for lab in range(1, n + 1):
        comp = labeled == lab
        if comp.sum() < MIN_AREA:
            continue
        refined = comp & (prob >= THR_HI)
        pieces = []
        if refined.sum() >= MIN_AREA:
            rl, rn = ndimage.label(refined.astype(np.uint8),
                                   structure=np.ones((3, 3), dtype=int))
            for r in range(1, rn + 1):
                m = rl == r
                if m.sum() >= MIN_AREA:
                    pieces.append(m)
        if not pieces:
            pieces = [comp]
        for m in pieces:
            pv = prob[m]
            score = 0.5 * float(pv.mean()) + 0.5 * float(np.percentile(pv, 95))
            out.append((m, score))
    out.sort(key=lambda t: -t[1])
    return out[:TOP_K]

def rle_encode(mask):
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    if isinstance(rle['counts'], bytes):
        rle['counts'] = rle['counts'].decode('ascii')
    return rle

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 4: LOAD ALL THREE ARMS
# ════════════════════════════════════════════════════════════════════════════

def load_v2_arm():
    models = []
    for name, path in V2_MODEL_LIST:
        cfg = V2_REGISTRY[name]
        m = getattr(smp, cfg["arch"])(encoder_name=cfg["encoder"],
                                      encoder_weights=None, in_channels=8,
                                      classes=1)
        sd = torch.load(path, map_location='cpu', weights_only=False)
        m.load_state_dict(sd['model_state_dict'] if 'model_state_dict' in sd
                          else sd)
        m.to(DEVICE).eval()
        models.append(m)
    return models

def load_v5c_arm():
    models = []
    for name in sorted(V5C_CKPTS):
        arch, enc = (("FPN", "mit_b5") if name == "fpn_mit_b5"
                     else ("Unet", "mit_b3"))
        m = getattr(smp, arch)(encoder_name=enc, encoder_weights=None,
                                in_channels=13, classes=1)
        sd = torch.load(V5C_CKPTS[name], map_location='cpu',
                        weights_only=False)
        m.load_state_dict(sd['model_state_dict'] if 'model_state_dict' in sd
                          else sd)
        m.to(DEVICE).eval()
        models.append(m)
    return models

def load_v5s_arm():
    """Run-1 harness winners: MAnet/mit_b4, ep100 + ep090 (arm-mean mirrors V5C)."""
    models = []
    for tag in V5S_TAGS:
        if tag not in V5S_CKPTS:
            continue
        m = getattr(smp, "MAnet")(encoder_name="mit_b4", encoder_weights=None,
                                   in_channels=13, classes=1)
        sd = torch.load(V5S_CKPTS[tag], map_location='cpu',
                        weights_only=False)
        m.load_state_dict(sd['model_state_dict'] if 'model_state_dict' in sd
                          else sd)
        m.to(DEVICE).eval()
        models.append(m)
    return models

print("[V5-E] Loading ARM A (V2) ...")
V2_MODELS = load_v2_arm()
print(f"[V5-E]   {len(V2_MODELS)} models loaded.")
print("[V5-E] Loading ARM B (V5C) ...")
V5C_MODELS = load_v5c_arm()
print(f"[V5-E]   {len(V5C_MODELS)} models loaded.")
print("[V5-E] Loading ARM C (V5S Run-1: MAnet/mit_b4 ep100+ep090) ...")
V5S_MODELS = load_v5s_arm()
print(f"[V5-E]   {len(V5S_MODELS)} checkpoints loaded.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 5: MERGED INFERENCE PER CHIP
# ════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def merged_prob(npz_path):
    """Each arm runs its OWN faithful path; merge on the native grid."""
    img8 = load_npz(npz_path)
    H, W = img8.shape[:2]

    prob_v2 = None
    if V2_MODELS:
        t = preprocess_v2(img8)
        pm = None
        for m in V2_MODELS:
            p = predict_v2(m, t)
            pm = p if pm is None else pm + p
        pm /= len(V2_MODELS)
        prob_v2 = cv2.resize(pm, (W, H), interpolation=cv2.INTER_LINEAR)

    prob_v5c = None
    if V5C_MODELS:
        x, native_hw = preprocess_v5c(img8)
        pm = None
        for m in V5C_MODELS:
            p = predict_v5c(m, x)[0, 0].numpy()
            pm = p if pm is None else pm + p
        pm /= len(V5C_MODELS)
        prob_v5c = prob_to_native(pm, native_hw)

    prob_v5s = None
    if V5S_MODELS:
        x, native_hw = preprocess_v5c(img8)      # same 13ch/512 path
        pm = None
        for m in V5S_MODELS:
            p = predict_v5c(m, x)[0, 0].numpy()  # same 8-fold TTA
            pm = p if pm is None else pm + p
        pm /= len(V5S_MODELS)
        prob_v5s = prob_to_native(pm, native_hw)

    # 3-arm weighted mean (renormalized over the arms actually present)
    arms = []
    if prob_v2 is not None:
        arms.append((W_V2, prob_v2))
    if prob_v5c is not None:
        arms.append((W_V5C, prob_v5c))
    if prob_v5s is not None:
        arms.append((W_V5S, prob_v5s))
    wsum = sum(w for w, _ in arms)
    prob = sum(w / wsum * p for w, p in arms)
    return prob, (H, W)

def resolve_test_path(base):
    path = os.path.join(TEST_IMG_DIR, base)
    if os.path.exists(path):
        return path
    hits = glob.glob(f"{DATA_DIR}/**/{base}", recursive=True)
    assert hits, f"test chip not found: {base}"
    return hits[0]

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 6: SUBMISSION + SELF-AUDIT
# ════════════════════════════════════════════════════════════════════════════

def generate_submission(out_path):
    manifest = pd.read_csv(TEST_MANIFEST)
    print(f"[V5-E] Test manifest: {len(manifest)} chips "
          f"(splits: {manifest['split'].value_counts().to_dict()})")
    preds = []
    t0 = time.time()
    for k, (_, row) in enumerate(manifest.iterrows()):
        img_id = int(row['image_id'])
        path = resolve_test_path(os.path.basename(row['file_name']))
        prob, _ = merged_prob(path)
        for m, score in instances_from_prob(prob):
            preds.append({"image_id": img_id, "category_id": 1,
                          "segmentation": rle_encode(m),
                          "score": round(score, 6)})
        if (k + 1) % 25 == 0:
            print(f"[V5-E]   test {k+1}/{len(manifest)} chips, "
                  f"{len(preds)} preds | {time.time()-t0:.0f}s")
    with open(out_path, 'w') as f:
        json.dump(preds, f)

    n_img = len({p['image_id'] for p in preds})
    scores = np.array([p['score'] for p in preds])
    areas = np.array([mask_utils.area(
        {**p['segmentation'],
         'counts': p['segmentation']['counts'].encode()}
        if isinstance(p['segmentation']['counts'], str) else p['segmentation'])
        for p in preds])
    audit = {
        "n_preds": len(preds), "images_with_preds": n_img,
        "preds_per_image": round(len(preds) / max(n_img, 1), 2),
        "score_min": float(scores.min()) if len(scores) else None,
        "score_median": float(np.median(scores)) if len(scores) else None,
        "score_max": float(scores.max()) if len(scores) else None,
        "area_p10": float(np.percentile(areas, 10)) if len(areas) else None,
        "area_median": float(np.median(areas)) if len(areas) else None,
        "area_p90": float(np.percentile(areas, 90)) if len(areas) else None,
        "share_lt_1024px": float((areas < 1024).mean()) if len(areas) else None,
    }
    print(f"[V5-E] SUBMISSION AUDIT: {json.dumps(audit, indent=2)}")
    print(f"[V5-E] GT density reference: 2.36/chip, expected ~326 preds")
    print(f"[V5-E] References: A (16.27 best) = 311 preds | B (16.08) = 309 | "
          f"C (15.73) = 312 | 2-arm 16.24 = 329 | V2=261 | V5C=306")
    return audit

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 7: MAIN
# ════════════════════════════════════════════════════════════════════════════

def main():
    sub_path = os.path.join(OUTPUT_DIR, "FrostWatch.json")
    audit = generate_submission(sub_path)
    with open(os.path.join(OUTPUT_DIR, "V5_PHASE_E_REPORT.json"), 'w') as f:
        json.dump({"phase": "E (Task 6, PRODUCTION = Submission A config)",
                   "arm_c": "V5S Run-1 ep100+ep090 (Run-2 swap tried as "
                            "Submission C: LB 15.73, DOWN -0.54, reverted)",
                   "fusion": FUSION, "w_v2": W_V2, "w_v5c": W_V5C,
                   "w_v5s": W_V5S, "n_v2_models": len(V2_MODELS),
                   "n_v5c_models": len(V5C_MODELS),
                   "n_v5s_ckpts": len(V5S_MODELS),
                   "v5s_run": 1,
                   "v5s_tags": [t for t in V5S_TAGS if t in V5S_CKPTS],
                   "submission_audit": audit}, f, indent=2)
    print(f"\n[V5-E] DONE. Submit {sub_path} to the HF space.")

if __name__ == "__main__":
    main()